In [ ]:
import numpy as np
import pandas as pd
import re
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
train_t = pd.read_csv('/content/sample_data/eng_french.csv')

In [ ]:
df = train_t[0:100000]

In [ ]:
english = []
french = []
for i in df['English words/sentences']:
    text = i.lower()
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = re.sub(' +', ' ', text)
    text = text[:-1]
    english.append(text)

for j in df['French words/sentences']:
    ftext =  j.lower()
    ftext = (re.sub("[^a-zA-Z' àâäèéêëîïôœùûüÿçÀÂÄÈÉÊËÎÏÔŒÙÛÜŸÇ]",' ',ftext))
    ftext = re.sub(' +',' ',ftext)
    ftext = ftext[:-1]
    french.append("start_ " + ftext + " _end")

In [ ]:
cleaned_data = pd.DataFrame({'english':english, 'french':french})
english_words = []
french_words = []

for i in cleaned_data['english']:
    english_words.append(len(i.split()))

for j in cleaned_data['french']:
    french_words.append(len(j.split()))

In [ ]:
sns.countplot(english_words)
plt.show()

sns.countplot(x=french_words)
plt.show()

In [ ]:
max_len_english = max(english_words)
max_len_french = max(french_words)

In [ ]:
from sklearn.model_selection import train_test_split
x_tr,x_val,y_tr,y_val=train_test_split(cleaned_data['english'],cleaned_data['french'],test_size=0.1,random_state=12,shuffle=True)

In [ ]:
from keras.preprocessing.text import Tokenizer
x_tokens = Tokenizer()
x_tokens.fit_on_texts(x_tr)

x_tr = x_tokens.texts_to_sequences(x_tr)
x_val = x_tokens.texts_to_sequences(x_val)

from keras.preprocessing.sequence import pad_sequences
x_tr = pad_sequences(x_tr,maxlen = max_len_english,padding = 'post')
x_val = pad_sequences(x_val,maxlen = max_len_english,padding = 'post')

# +1 for padding 
x_voc_size   =  len(x_tokens.word_index) +1

# y data
from keras.preprocessing.text import Tokenizer
y_tokens = Tokenizer()
y_tokens.fit_on_texts(y_tr)

y_tr = y_tokens.texts_to_sequences(y_tr)
y_val = y_tokens.texts_to_sequences(y_val)

from keras.preprocessing.sequence import pad_sequences
y_tr = pad_sequences(y_tr,maxlen = max_len_french,padding = 'post')
y_val = pad_sequences(y_val,maxlen = max_len_french,padding = 'post')

# +1 for padding 
y_voc_size   =  len(y_tokens.word_index) +1

In [ ]:
from attention import AttentionLayer
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Concatenate, TimeDistributed, Bidirectional
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from keras import backend as K 
K.clear_session()

latent_dim = 300

encoder_inputs = Input(shape=(max_len_english,))

enc_emb =  Embedding(x_voc_size, latent_dim,trainable=True)(encoder_inputs)

encoder_lstm1 = Bidirectional(LSTM(latent_dim,return_sequences=True,return_state=True))
encoder_output1, forward_h1, forward_c1, backward_h1, backward_c1 = encoder_lstm1(enc_emb)

encoder_lstm2 = Bidirectional(LSTM(latent_dim,return_sequences=True,return_state=True))
encoder_output2, forward_h2, forward_c2, backward_h2, backward_c2 = encoder_lstm2(encoder_output1)

encoder_lstm3 = Bidirectional(LSTM(latent_dim, return_state=True, return_sequences=True))
encoder_outputs, forward_h3, forward_c3, backward_h3, backward_c3= encoder_lstm3(encoder_output2)

state_h = Concatenate()([forward_h3,backward_h3])
state_c = Concatenate()([forward_c3,backward_c3])

decoder_inputs = Input(shape=(None,))

dec_emb_layer = Embedding(y_voc_size, latent_dim,trainable=True)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(600, return_sequences=True, return_state=True)
decoder_outputs,decoder_fwd_state, decoder_back_state = decoder_lstm(dec_emb,initial_state=[state_h, state_c])

attn_layer = AttentionLayer(name='attention_layer')
attn_out, attn_states = attn_layer([encoder_outputs, decoder_outputs])

decoder_concat_input = Concatenate(axis=-1, name='concat_layer')([decoder_outputs, attn_out])

decoder_dense =  TimeDistributed(Dense(y_voc_size, activation='softmax'))
decoder_out = decoder_dense(decoder_concat_input)

model = Model([encoder_inputs, decoder_inputs], decoder_out)

model.summary()


In [ ]:
def merge_sort(arr):
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2

    left = arr[:mid]
    right = arr[mid:]
    print(left)
    print(right)
    left = merge_sort(left)
    right = merge_sort(right)
    print(f'**{left}**')
    print(f'***{right}***')
    return merge_two_sorted_list(left, right)

In [ ]:
arr=[10, 3, 15, 7, 8, 23, 98, 29]
merge_sort(arr)

[10, 3, 15, 7]
[8, 23, 98, 29]
[10, 3]
[15, 7]
[10]
[3]
**[10]**
***[3]***
[15]
[7]
**[15]**
***[7]***
**[3, 10]**
***[7, 15]***
[8, 23]
[98, 29]
[8]
[23]
**[8]**
***[23]***
[98]
[29]
**[98]**
***[29]***
**[8, 23]**
***[29, 98]***
**[3, 7, 10, 15]**
***[8, 23, 29, 98]***


[3, 7, 8, 10, 15, 23, 29, 98]

In [ ]:
def merge_two_sorted_list(a, b):
    sorted_list = []
    len_a = len(a)
    len_b = len(b)

    i = j = 0

    while i < len_a and j < len_b:
        if a[i] <= b[j]:
            sorted_list.append(a[i])
            i += 1

        else:
            sorted_list.append(b[j])
            j += 1

    while i < len_a:
        sorted_list.append(a[i])
        i += 1

    while j < len_b:
        sorted_list.append(b[j])
        j += 1

    return sorted_list

In [ ]:
def twoSum(arr, target):
    for i in range(len(arr)):
        for j in range(i+1, len(arr)):
            if arr[j] == target - arr[i]:
                return [i, j]

twoSum([1,2,5,7], 7)

[1, 2]

In [ ]:
l1 = [2, 4, 3]
l2 = [5, 6, 4]
for i in reversed(l1):
    print(i)


3
4
2


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

dummy = ListNode()

In [ ]:
st = 'abcdefgha'
chars = [0] * 128
for i in range(len(st)):
    c = st[i]
    chars[ord(c)] += 1

print(chars)


    

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
def isPalindrom(s):
    if s is "":
        return False

    for i in range(len(s)//2):
        if s[i] != s[-1-i]:
            return False
    return True


In [ ]:
class Solution2:
    def longestPalindrome(self, s: str):
        def isPalindrome(s):
            if s is "":
                return False
            for i in range(len(s)//2):
                if s[i] != s[-1-i]:
                    return False
            return True
        common_subs = {}
        
        if s is "":
            return ""
        res = 0   
        for i in range(len(s)):
            for j in range(1, len(s)+1):
                if isPalindrome(s[i:j]):
                    common_subs[s[i:j]] = len(s[i:j])
            
        
        return max(common_subs, key=common_subs.get)

In [ ]:
pl = Solution2()
pl.longestPalindrome("babbabc")

'babbab'

In [ ]:
def longestPalin(s):
    res = ""
    resLen = 0

    for i in range(len(s)):
        # For odd String
        l, r = i, i
        while l >= 0 and r < len(s) and s[l] == s[r]:
            if (r - l + 1) > resLen:
                res = s[l:r+1]
                resLen = r - l + 1
            l -= 1
            r += 1

        # For Even string
        l, r = i, i + 1
        while l >= 0 and r < len(s) and s[l] == s[r]:
            if (r - l + 1) > resLen:
                res = s[l:r+1]
                resLen = r - l + 1
            l -= 1
            r += 1

    return resLen

In [ ]:
def zigZag(s, rows):
    if rows == 1:
        return s

    res = ""

    for r in range(rows):
        increment = 2 * (rows - 1)
        for i in range(r, len(s), increment):
            res += s[i]

        # for middle rows
            if r > 0 and r < rows - 1 and i + increment - 2 * r < len(s):
                res += s[i + increment - 2 * r]

    return res


    

In [ ]:
zigZag('PAYPALISHIRING', 3)

'PAHNAPLSIIGYIR'

In [ ]:
def reverse(x):
    MIN = -2147483648   # -2^31
    MAX = 2147483647    # -2^31 - 1

    res = 0
    while x:
        digit = x % 10
        x = int(x / 10)

        if (res > MAX // 10 or (res == MAX // 10 and digit >= MAX % 10) ):
            return 0

        if (res < MIN // 10 or (res == MIN // 10 and digit <= MIN % 10)):
            return 0

        res = (res * 10) + digit

    return res

In [ ]:
reverse(145678)

876541

In [ ]:
def intPalin(x):
    if x < 0 or (x > 10 and x % 10 == 0):
        return False
    res = 0
    while x > res:
        res = res * 10 + x % 10
        x = x // 10
    return True if (x == res or x == res // 10) else False

In [ ]:
def maxArea(height):
    res = 0
    for i in range(len(height)):
        for j in range(i+1, len(height)):
            area = (j - i) * min(height[i], height[j])
            res = max(res, area)
    return res
            


In [ ]:
def intToRoman(num):
    values = [(1000, "M"), (900, "CM"), (500, "D"), (400, "CD"), (100, "C"),
              (90, "XC"), (50, "L"), (40, "XL"), (10, "X"), (9, "IX"),
              (5, "V"), (4, "IV"), (1, "I")]
    roman_digits = []
    for value, symbol in values:
        if num == 0:
            break
        count, num = divmod(num, value)
        print(symbol, count)
        roman_digits.append(symbol * count)
        

    return "".join(roman_digits)
    

In [ ]:
values = {"I":1, "V":5, "X":10, "L":50, "C":100, "D":500, "M":1000}
def romanToInt(st):
    i = 0
    total = 0
    while i < len(st):
        if i+1 < len(st) and values[st[i]] < values[st[i+1]]:
            total += values[st[i+1]] - values[st[i]]
            i += 2
        else:
            total += values[st[i]]
            i += 1
    return total

In [ ]:
def longestCommonPrefix(st):
    res = ""
    for i in zip(*st):
        print(i)
    return True

In [38]:
def commomLongestPrefix(x):
    prefix=[]
    num = len(x)
    for i in zip(*x):
        if len(set(i)) == 1:
            prefix.append(i[0])
        else:
            break
    return "".join(prefix)

x='flower'
print(list(zip(*x)))

[('f', 'l', 'o', 'w', 'e', 'r')]


In [ ]:
x=["flower","flow","flight"]
print(list(zip(*x)))

[('f', 'f', 'f'), ('l', 'l', 'l'), ('o', 'o', 'i'), ('w', 'w', 'g')]


In [ ]:
commomLongestPrefix(["flower","flow","flight"])

In [ ]:
longestCommonPrefix(["flower","flow","flight"])


In [ ]:
def threeSum(arr):
    res=[]
    if len(arr) == 0 or len(arr)==1:
        return arr
    for i in range(len(arr)):
        for j in range(i+1, len(arr)):
            for k in range(i+2, len(arr)):
                if (i != j) and (i != k) and (j != k) and i+j+k == 0:
                    res.append([i,j,k])
    return res


In [ ]:
threeSum([-1,0,1,2,-1,-4])

[]

In [ ]:
#Leet Code 167 - Two Sum II
def twoSumII(numbers, target):
    l,r = 0, len(numbers) - 1
    while l < r:
        curSum = numbers[l] + numbers[r]
        if curSum > target:
            r -= 1
        elif curSum < target:
            l += 1
        else:
            return (l+1, r+1)

In [ ]:
twoSumII([2,7,11,15], 9)

In [ ]:
def threeSumCloset(nums, target):
    nums.sort()
    diff = float('inf')
    for i in range(len(nums)):
        l,r = i+1, len(nums)-1
        while (l < r):
            threeSum = nums[i] + nums[l] + nums[r]
            if abs(target - threeSum ) < abs(diff):
                diff = target - sum
            if threeSum < target:
                l += 1
            else:
                r -= 1
        if diff == 0:
            break
    return target - diff




In [1]:
def listPosition(arr, target):
    hash={}
    for i in range(len(arr)):
        if target in hash:
            return hash[target] 
        hash[arr[i]] = i
    return hash[target]
        


In [3]:
listPosition([1,2,3,5,7], 5)

3

In [24]:
def pen(s):
    #if s != ''.join(reversed(s)):
    if s == s[::-1]:
        return True
    return False

x='abcdef'
print(list(reversed(x)))
    

In [26]:
pen('abcba')

True

In [ ]:
#output = [My name, name is, is AA, AA and .... ]
text = "my name is aa and I am python developer"
wordList = text.split()
twogram = []
for i in range(len(wordList)-1):
    w=wordList[i] + ' ' + wordList[i+1]
    print(w)
    twogram.append(w)


In [ ]:
l1 = 'my'
l2 = 'name is'
l3 = 'aaa and python'

#res = l1 + ' ' + l2 + ' ' + l3

res = (l1 + ' ' + l2 + ' ' + l3).split()
finalList = []
for i in range(len(res)):
    finalList.append(res[i].capitalize())




In [17]:
ll = [0]*24
st = 'aabbbxe'
for i in range(len(st)):
    x=ord(st[i])
    print(x)



97
97
98
98
98
120
101
